In [33]:
import numpy as np
import pandas as pd
import duckdb
import requests
from dotenv import load_dotenv
import os


In [34]:
url = 'https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/users.parquet'

In [35]:
duckdb.__version__

'1.3.0'

In [36]:
conn = duckdb.connect()
conn.execute(f"SELECT * FROM '{url}' LIMIT 5").df()

,id,first_name,last_name,phone,sex,birth_date
0,1,Ульяна,None,+70694750861,F,2000-06-22
1,2,Юлия,Пахомова,+71980399641,F,2004-11-21
2,3,Нифонт,None,+72671331267,M,1998-02-06
3,4,Герман,Филатов,+74697796583,M,2003-06-07
4,5,Ким,None,+71443136825,M,1989-10-10


In [37]:
conn.execute('SELECT version()').df()

,"""version""()"
0,v1.3.0


In [38]:
conn.execute('SELECT 1+1, sqrt(16), pow(2, 8), 1/0').df()

,(1 + 1),sqrt(16),"pow(2, 8)",(1 / 0)
0,2,4.0,256.0,inf


In [39]:
csv_df = 'supporters.csv'

In [40]:
df = conn.execute(f'SELECT * from {csv_df}').df()

In [41]:
df.merge(df, how='left',on='id')

,id,name_x,birthday_x,sex_x,amount_rub_x,is_regular_x,name_y,birthday_y,sex_y,amount_rub_y,is_regular_y
0,1,Антон,2004-01-24,M,1000.0,True,Антон,2004-01-24,M,1000.0,True
1,2,Олег,1974-04-01,M,10.0,False,Олег,1974-04-01,M,10.0,False
2,4,Марк,1998-08-23,M,150.0,True,Марк,1998-08-23,M,150.0,True
3,5,Мама,1980-02-21,F,15000.0,True,Мама,1980-02-21,F,15000.0,True
4,6,Михаил,1995-11-18,M,100.0,False,Михаил,1995-11-18,M,100.0,False
5,7,Наталья,1994-10-02,F,750.0,True,Наталья,1994-10-02,F,750.0,True
6,8,Сергей,2002-05-19,M,1.0,False,Сергей,2002-05-19,M,1.0,False
7,9,Алексей,2003-04-06,M,950.0,False,Алексей,2003-04-06,M,950.0,False
8,10,Владимир,2008-08-21,M,300.0,False,Владимир,2008-08-21,M,300.0,False
9,11,Катерина,1983-03-14,F,200.0,True,Катерина,1983-03-14,F,200.0,True


In [42]:
conn.execute('''
-- Построим гистограмму по возрасту спонсоров
-- SELECT * в DuckDB можно опустить
FROM histogram('supporters.csv', datediff('year', birthday, current_date))
''').df()

,bin,count,bar
0,x <= 20,1,████████████████████ ...
1,20 < x <= 25,4,██████████████████████████████████████████████...
2,25 < x <= 30,1,████████████████████ ...
3,30 < x <= 35,2,████████████████████████████████████████ ...
4,35 < x <= 40,1,████████████████████ ...
5,40 < x <= 45,1,████████████████████ ...
6,45 < x <= 50,1,████████████████████ ...
7,50 < x <= 55,1,████████████████████ ...


In [43]:
conn.execute('''
PIVOT (
    SELECT
        amount_rub,
        is_regular
    FROM df
)
ON is_regular
USING
    sum(amount_rub),
    median(amount_rub)
''').df()

,false_sum(amount_rub),false_median(amount_rub),true_sum(amount_rub),true_median(amount_rub)
0,1876.0,100.0,17100.0,750.0


In [44]:
conn.execute('''
PIVOT (
    SELECT
        amount_rub,
        is_regular
    FROM df
)
ON is_regular
USING
    sum(amount_rub),
    median(amount_rub)
''').df()


,false_sum(amount_rub),false_median(amount_rub),true_sum(amount_rub),true_median(amount_rub)
0,1876.0,100.0,17100.0,750.0


In [45]:
conn.execute('''CREATE TABLE trips AS 
SELECT * FROM 'trips.csv';''')


In [46]:
conn.execute('select * from trips limit 5').df()

,id,user_id,scooter_hw_id,started_at,finished_at,distance,price
0,1,1393,GT-GXLV2-99D4,2023-06-01 06:06:02+03:00,2023-06-01 06:24:49+03:00,1644.987,0
1,2,1987,XM-1S-56D8X,2023-06-01 06:27:24+03:00,2023-06-01 06:53:51+03:00,4430.513,0
2,3,981,SN-ES2-82XF4,2023-06-01 06:32:32+03:00,2023-06-01 07:05:32+03:00,4473.653,0
3,4,1629,SN-MAXG30-56D2G,2023-06-01 06:36:45+03:00,2023-06-01 07:00:30+03:00,2506.703,23750
4,5,602,SN-MAXG30-56D2G,2023-06-01 06:42:08+03:00,2023-06-01 07:10:24+03:00,4883.543,0


In [47]:
scooters = 'scooters.csv'
conn.execute(f'create table scooters as select * from {scooters}')

In [48]:
conn.execute('select * from scooters limit 5').df()

,company,model,hardware_id,scooters
0,GoTrax,GXL V2,GT-GXLV2-99D4,440
1,Segway-Ninebot,KickScooter ES2,SN-ES2-82XF4,447
2,Segway-Ninebot,KickScooter Max G30,SN-MAXG30-56D2G,489
3,Segway-Ninebot,Ninebot Air T15,SN-AIRT15-84JK9,431
4,Skip,Skip S3,SKP-S3-4FX87,445


In [49]:
#Надо добавить к данным из trips столбец company из scooters, результат записать в trips_company
conn.execute(''' create view agg as select t.*, s.company from trips t 
left join scooters s on s.hardware_id = t.scooter_hw_id''')

In [50]:
conn.execute('select * from agg limit 5').df()

,id,user_id,scooter_hw_id,started_at,finished_at,distance,price,company
0,1,1393,GT-GXLV2-99D4,2023-06-01 06:06:02+03:00,2023-06-01 06:24:49+03:00,1644.987,0,GoTrax
1,2,1987,XM-1S-56D8X,2023-06-01 06:27:24+03:00,2023-06-01 06:53:51+03:00,4430.513,0,Xiaomi
2,3,981,SN-ES2-82XF4,2023-06-01 06:32:32+03:00,2023-06-01 07:05:32+03:00,4473.653,0,Segway-Ninebot
3,4,1629,SN-MAXG30-56D2G,2023-06-01 06:36:45+03:00,2023-06-01 07:00:30+03:00,2506.703,23750,Segway-Ninebot
4,5,602,SN-MAXG30-56D2G,2023-06-01 06:42:08+03:00,2023-06-01 07:10:24+03:00,4883.543,0,Segway-Ninebot


In [51]:
#3. Надо посчитать среднюю цену поездки в рублях price_rub_avg для каждой компании-производителя (company)

conn.execute('select company, avg(price) price_rub_avg from agg \
group by 1 order by 2 desc').df()

,company,price_rub_avg
0,GoTrax,10944.333333
1,Skip,9977.090909
2,Xiaomi,9868.100000
3,Spin,9526.600000
4,Segway-Ninebot,9516.562500
5,Unagi,0.000000


In [52]:
trips_full = 'https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/trips.parquet'
conn.execute(f"""create table trips_full as select * from '{trips_full}'""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [53]:
conn.execute('select * from trips_full limit 10').df()

,id,user_id,scooter_hw_id,started_at,finished_at,start_lat,start_lon,finish_lat,finish_lon,distance,price
0,1,1393,GT-GXLV2-99D4,2023-06-01 06:06:02+03:00,2023-06-01 06:24:49+03:00,55.760750,37.613998,55.771710,37.618098,1644.987,0
1,2,1987,XM-1S-56D8X,2023-06-01 06:27:24+03:00,2023-06-01 06:53:51+03:00,55.742681,37.651062,55.737762,37.601283,4430.513,0
2,3,981,SN-ES2-82XF4,2023-06-01 06:32:32+03:00,2023-06-01 07:05:32+03:00,55.761562,37.642959,55.749707,37.589230,4473.653,0
3,4,1629,SN-MAXG30-56D2G,2023-06-01 06:36:45+03:00,2023-06-01 07:00:30+03:00,55.743430,37.590626,55.735752,37.614239,2506.703,23750
4,5,602,SN-MAXG30-56D2G,2023-06-01 06:42:08+03:00,2023-06-01 07:10:24+03:00,55.731653,37.623907,55.769428,37.623847,4883.543,0
5,6,659,SN-ES2-82XF4,2023-06-01 06:45:30+03:00,2023-06-01 07:04:58+03:00,55.764658,37.627241,55.745587,37.635353,2963.492,0
6,7,1330,SN-MAXG30-56D2G,2023-06-01 06:53:24+03:00,2023-06-01 07:24:01+03:00,55.732403,37.665843,55.760160,37.622423,4630.590,0
7,8,1312,SKP-S3-4FX87,2023-06-01 06:57:24+03:00,2023-06-01 07:23:19+03:00,55.759995,37.639000,55.736872,37.636059,3171.365,25916
8,9,1614,XM-1S-56D8X,2023-06-01 07:03:30+03:00,2023-06-01 07:21:44+03:00,55.763360,37.626385,55.744701,37.643575,2787.975,18233
9,10,1547,XM-PRO2-7743,2023-06-01 07:03:38+03:00,2023-06-01 07:08:18+03:00,55.762369,37.639876,55.766475,37.640796,723.596,0


In [54]:
conn.execute(''' create view if not exists agg2 as select t.*, s.company from trips_full t 
left join scooters s on s.hardware_id = t.scooter_hw_id''')
conn.execute('select company, avg(price) price_rub_avg, count(1) as trips_cnt from agg2 \
group by 1 order by 2 desc').df()

,company,price_rub_avg,trips_cnt
0,Skip,22644.509865,10289
1,Unagi,22286.691249,11187
2,Spin,22274.751281,10928
3,Xiaomi,22046.538641,30822
4,Segway-Ninebot,21940.967154,32241
5,GoTrax,21676.598524,10434


In [ ]:
weather = 'https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/weather.json'
conn.execute(f"""drop table if exists weather""")
conn.execute(f"""create table weather as select * from read_json('{weather}', dateformat='%Y-%m-%dT%H:%M:%S.%f')""")#для парсинга дат

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [63]:
conn.execute('select * from weather').df()

,date,condition
0,2023-06-01,sun
1,2023-06-02,clouds
2,2023-06-03,sun
3,2023-06-04,sun
4,2023-06-05,sun
...,...,...
87,2023-08-27,sun
88,2023-08-28,sun
89,2023-08-29,sun
90,2023-08-30,clouds


In [72]:
#легкое объединение разнородных датасетов
conn.execute("""CREATE OR REPLACE VIEW trips_weather AS
SELECT
    t.*,
    w."condition"
FROM
    trips_full AS t
    LEFT JOIN weather AS w
      ON t.started_at::DATE = w.date""")

In [73]:
conn.execute('select * from trips_weather').df().head(5)

,id,user_id,scooter_hw_id,started_at,finished_at,start_lat,start_lon,finish_lat,finish_lon,distance,price,condition
0,1,1393,GT-GXLV2-99D4,2023-06-01 06:06:02+03:00,2023-06-01 06:24:49+03:00,55.760750,37.613998,55.771710,37.618098,1644.987,0,sun
1,2,1987,XM-1S-56D8X,2023-06-01 06:27:24+03:00,2023-06-01 06:53:51+03:00,55.742681,37.651062,55.737762,37.601283,4430.513,0,sun
2,3,981,SN-ES2-82XF4,2023-06-01 06:32:32+03:00,2023-06-01 07:05:32+03:00,55.761562,37.642959,55.749707,37.589230,4473.653,0,sun
3,4,1629,SN-MAXG30-56D2G,2023-06-01 06:36:45+03:00,2023-06-01 07:00:30+03:00,55.743430,37.590626,55.735752,37.614239,2506.703,23750,sun
4,5,602,SN-MAXG30-56D2G,2023-06-01 06:42:08+03:00,2023-06-01 07:10:24+03:00,55.731653,37.623907,55.769428,37.623847,4883.543,0,sun


In [74]:
conn.execute("""SELECT
    "condition",
    median(distance) / 1000 as median_distance_km
FROM trips_weather
GROUP BY ALL
ORDER BY ALL""").df()

,condition,median_distance_km
0,clouds,3.293665
1,rain,3.292475
2,sun,3.297972


In [71]:
conn.execute("""PIVOT (
    SELECT
        "condition",
        distance / 1000 as distance_km,
    FROM trips_weather
)
USING median(distance_km) as median_distance_km
GROUP BY condition""").df()

,condition,median_distance_km
0,sun,3.414977


Подключаемся к базе данных через DuckDB (для аналитики без инджестинга в базу):

In [ ]:
# Загружаем данные из .env файла для параметров подключения к бд
load_dotenv()
# Получаем значение переменной
user = os.getenv("DWH_USER")
password = os.getenv("DWH_PASSWORD")
db = os.getenv("DWH_DB_NAME")
#print(token)

In [ ]:
#создаем подключение
#https://duckdb.org/docs/current/core_extensions/postgres/overview
conn.execute(f"ATTACH 'dbname={db} user={user} host=localhost password={password}' AS db (TYPE postgres, SCHEMA 'raw');")


DETAIL:  The database was created using collation version 2.36, but the operating system provides version 2.31.
HINT:  Rebuild all objects in this database that use the default collation and run ALTER DATABASE dwh REFRESH COLLATION VERSION, or build PostgreSQL with the right library version.


In [ ]:
#выполняем запрос к созданной в проекте dbt модели
conn.execute('SELECT * FROM db.events_clean_v2').df()

DETAIL:  The database was created using collation version 2.36, but the operating system provides version 2.31.
HINT:  Rebuild all objects in this database that use the default collation and run ALTER DATABASE dwh REFRESH COLLATION VERSION, or build PostgreSQL with the right library version.
DETAIL:  The database was created using collation version 2.36, but the operating system provides version 2.31.
HINT:  Rebuild all objects in this database that use the default collation and run ALTER DATABASE dwh REFRESH COLLATION VERSION, or build PostgreSQL with the right library version.


,user_id,timestamp,type_id,updated_at,date
0,1,2023-08-30 08:55:22+03:00,2,2026-10-08 20:50:29.825239+03:00,2023-08-30
1,10,2023-08-30 08:16:20+03:00,1,2026-10-08 20:50:29.825239+03:00,2023-08-30
2,36,2023-08-30 18:55:48+03:00,0,2026-10-08 20:50:29.825239+03:00,2023-08-30
3,36,2023-08-30 19:47:52+03:00,2,2026-10-08 20:50:29.825239+03:00,2023-08-30
4,59,2023-08-30 18:18:24+03:00,0,2026-10-08 20:50:29.825239+03:00,2023-08-30
...,...,...,...,...,...
322742,1800,2023-08-30 08:44:24+03:00,3,2026-10-08 20:50:29.825239+03:00,2023-08-30
322743,1800,2023-08-30 18:00:47+03:00,0,2026-10-08 20:50:29.825239+03:00,2023-08-30
322744,1802,2023-08-30 09:56:12+03:00,1,2026-10-08 20:50:29.825239+03:00,2023-08-30
322745,1809,2023-08-30 08:44:31+03:00,1,2026-10-08 20:50:29.825239+03:00,2023-08-30


In [ ]:
#связываем данные о погоде из файла с таблицей о выручке в БД
conn.execute("""SELECT
    avg(revenue_rub) as revenue_rub_avg,
    "condition"
FROM
    db.revenue_daily AS r
    LEFT JOIN weather AS w USING(date)
GROUP BY ALL
ORDER BY ALL""").df()

,revenue_rub_avg,condition
0,251268.481538,rain
1,253473.886727,sun
2,268735.200870,clouds


In [80]:
#сохраняем результат запроса для отправки
conn.execute("""COPY (
    SELECT
        avg(revenue_rub) as revenue_rub_avg,
        "condition"
    FROM
        db.revenue_daily AS r
        LEFT JOIN weather AS w USING(date)
    GROUP BY ALL
    ORDER BY ALL
)
TO 'revenue_weather.csv'""")